In [1]:
import duckdb

### 1. Download Data

Done using the script/download_trains_data.rb

### 2. Put stations data into stations table in DuckDB. This changes rarely, so we treat it as a almost constant file.

In [2]:
with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql("""
        CREATE TABLE IF NOT EXISTS stations AS
        FROM "data/stations-2023-09.csv"
    """)
    print("stations imported to data/duckdb_trains.db")

    display(db.sql("FROM stations SELECT COUNT(*) AS stations_count"))

stations imported to data/duckdb_trains.db


┌────────────────┐
│ stations_count │
│     int64      │
├────────────────┤
│            591 │
└────────────────┘

### 3. Based on DuckDB tutorial, create tables distances and distances_long. We treat this similarly to stations table.

In [3]:
!head -n 9 data/tariff-distances-2022-01.csv | cut -d, -f1-9

Station,AC,AH,AHP,AHPR,AHZ,AKL,AKM,ALM
AC,XXX,82,83,85,90,71,188,32
AH,82,XXX,1,3,8,77,153,98
AHP,83,1,XXX,2,9,78,152,99
AHPR,85,3,2,XXX,11,80,150,101
AHZ,90,8,9,11,XXX,69,161,106
AKL,71,77,78,80,69,XXX,211,96
AKM,188,153,152,150,161,211,XXX,158
ALM,32,98,99,101,106,96,158,XXX


In [4]:
with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql("""
        CREATE TABLE IF NOT EXISTS distances AS
        FROM read_csv('data/tariff-distances-2022-01.csv', nullstr = 'XXX')
    """)

    display(db.sql("FROM (DESCRIBE distances) LIMIT 5;"))
    display(db.sql("FROM distances SELECT COUNT(*) AS distances_count"))
    display(db.sql(
        """SELECT #1, #2, #3, #4, #5, #6, #7, #8, #9
            FROM distances
            LIMIT 8;
        """))



┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Station     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ AC          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ AH          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ AHP         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ AHPR        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────────────┐
│ distances_count │
│      int64      │
├─────────────────┤
│             399 │
└─────────────────┘

┌─────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ Station │  AC   │  AH   │  AHP  │ AHPR  │  AHZ  │  AKL  │  AKM  │  ALM  │
│ varchar │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │
├─────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┤
│ AC      │  NULL │    82 │    83 │    85 │    90 │    71 │   188 │    32 │
│ AH      │    82 │  NULL │     1 │     3 │     8 │    77 │   153 │    98 │
│ AHP     │    83 │     1 │  NULL │     2 │     9 │    78 │   152 │    99 │
│ AHPR    │    85 │     3 │     2 │  NULL │    11 │    80 │   150 │   101 │
│ AHZ     │    90 │     8 │     9 │    11 │  NULL │    69 │   161 │   106 │
│ AKL     │    71 │    77 │    78 │    80 │    69 │  NULL │   211 │    96 │
│ AKM     │   188 │   153 │   152 │   150 │   161 │   211 │  NULL │   158 │
│ ALM     │    32 │    98 │    99 │   101 │   106 │    96 │   158 │  NULL │
└─────────┴───────┴───────┴───────┴───────┴───────┴───────┴───────┴───────┘

In [5]:
with duckdb.connect("data/duckdb_trains.db") as db:
    display(db.sql("""
        CREATE TABLE IF NOT EXISTS distances_long AS
            UNPIVOT distances
            ON COLUMNS (* EXCLUDE station)
            INTO NAME other_station VALUE distance;
    """))
    display(db.sql("""
        SELECT station, other_station, distance
            FROM distances_long
            LIMIT 5;
    """))
    display(db.sql("""
        SELECT
            s1.name_long AS station1,
            s2.name_long AS station2,
            distances_long.distance
        FROM distances_long
        JOIN stations s1 ON distances_long.station = s1.code
        JOIN stations s2 ON distances_long.other_station = s2.code
        WHERE s1.country = 'NL'
          AND s2.country = 'NL'
          AND station < other_station
        ORDER BY distance DESC
        LIMIT 5;
    """))





None

┌─────────┬───────────────┬──────────┐
│ Station │ other_station │ distance │
│ varchar │    varchar    │  int64   │
├─────────┼───────────────┼──────────┤
│ AC      │ AH            │       82 │
│ AC      │ AHP           │       83 │
│ AC      │ AHPR          │       85 │
│ AC      │ AHZ           │       90 │
│ AC      │ AKL           │       71 │
└─────────┴───────────────┴──────────┘

┌──────────────────┬────────────────────┬──────────┐
│     station1     │      station2      │ distance │
│     varchar      │      varchar       │  int64   │
├──────────────────┼────────────────────┼──────────┤
│ Eemshaven        │ Vlissingen         │      426 │
│ Eemshaven        │ Vlissingen Souburg │      425 │
│ Bad Nieuweschans │ Vlissingen         │      425 │
│ Bad Nieuweschans │ Vlissingen Souburg │      424 │
│ Stavoren         │ Vlissingen         │      421 │
└──────────────────┴────────────────────┴──────────┘

### 4. Put train disruptions into disruptions table in the Postgres database. We expect this data to change regularly, and thus treat it as a typical OLTP table.

In [6]:
conn_string = "host=localhost user=postgres password=postgres dbname=postgres"

# Create read-write connection to Postgres
with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql(f"""
    ATTACH IF NOT EXISTS '{conn_string}' AS postgres_db (TYPE postgres);
    """)

    # load 2011-2023 disruptions into Postgres
    db.sql("""
        CREATE OR REPLACE TABLE postgres_db.disruptions AS
            FROM read_csv('data/disruptions-*.csv')
    """)


    display(db.sql("""
        SELECT
            YEAR(start_time) AS year,
            COUNT(*) AS num_disruptions
        FROM postgres_db.disruptions
        WHERE YEAR(start_time) BETWEEN 2011 AND 2023
        GROUP BY YEAR(start_time)
        ORDER BY year
    """))



┌───────┬─────────────────┐
│ year  │ num_disruptions │
│ int64 │      int64      │
├───────┼─────────────────┤
│  2011 │            1846 │
│  2012 │            2074 │
│  2013 │            2312 │
│  2014 │            2484 │
│  2015 │            2947 │
│  2016 │            3031 │
│  2017 │            4085 │
│  2018 │            5190 │
│  2019 │            5940 │
│  2020 │            4450 │
│  2021 │            4874 │
│  2022 │            5499 │
│  2023 │            5168 │
├───────┴─────────────────┤
│ 13 rows       2 columns │
└─────────────────────────┘

### 5. Transform train services CSV files into a single Parquet file. Make table services from it. We treat this as a big data batch input, created rarely but regularly for analytics purposes.
> We treat this as a big data batch input, created rarely but regularly for analytics purposes.

We want to store it in duckdb.

In [7]:
# 5.1.  lets transform services 2019-2025 CSVs into Parquet file
with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql(f"""
        COPY (
            SELECT * FROM read_csv('data/services-20??.csv', union_by_name=True)
        ) TO 'data/services_2019_2025.parquet' (FORMAT PARQUET);
    """)

    print(db.sql("SELECT count(*) FROM 'data/services_2019_2025.parquet'").fetchall())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(152176324,)]


In [8]:
with duckdb.connect("data/duckdb_trains.db") as db:
    display(db.sql("SELECT * FROM 'data/services_2019_2025.parquet' LIMIT 5;"))
    display(db.sql("""SELECT YEAR("Service:Date"), count(*) FROM 'data/services_2019_2025.parquet' GROUP BY YEAR("Service:Date")""").fetchall())

┌────────────────┬──────────────┬──────────────┬─────────────────┬──────────────────────┬──────────────────────────────┬──────────────────────────┬───────────────────────┬─────────────┬───────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────────┬─────────────────────┬──────────────────────┬──────────────────────────┬──────────────────────┬───────────────────────┬──────────────────────┐
│ Service:RDT-ID │ Service:Date │ Service:Type │ Service:Company │ Service:Train number │ Service:Completely cancelled │ Service:Partly cancelled │ Service:Maximum delay │ Stop:RDT-ID │ Stop:Station code │ Stop:Station name  │  Stop:Arrival time  │ Stop:Arrival delay │ Stop:Arrival cancelled │ Stop:Departure time │ Stop:Departure delay │ Stop:Departure cancelled │ Stop:Platform change │ Stop:Planned platform │ Stop:Actual platform │
│     int64      │     date     │   varchar    │     varchar     │        int64         │           boolean            │      

[(2019, 20773804),
 (2020, 22235137),
 (2021, 21818011),
 (2022, 22025754),
 (2023, 21239393),
 (2024, 21857914),
 (2025, 22226311)]

In [24]:
# 5.2. Create `services` table in duckdb from the parquet file. In DuckDB as it is analytics table.

with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql("""
    CREATE TABLE IF NOT EXISTS services AS
    FROM "data/services_2019_2025.parquet"
    """)

    services_description = db.sql("DESCRIBE services").show()
    services_count = db.sql("SELECT count(*) AS services_count FROM services").fetchone()

print(f"services table {services_count=}")
display(services_description)

services_count

┌──────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │ column_type │  null   │   key   │ default │  extra  │
│           varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Service:RDT-ID               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Date                 │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Type                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Company              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Train number         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Completely cancelled │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Partly cancelled     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Maximum delay        │ BIGINT    

None

(21857914,)

### 6. In all following questions, use tables services, distances_long, disruptions and stations as necessary. Remember that each row represents single ride between stations, while the whole end-to-end train service has the same value in Service:RDT-ID column.

Note that:
 - starting station for a service has NULL value in `Stop:Arrival` time column,
 - while the end station has NULL value in `Stop:Departure` time column.

In [29]:
# tables to use

with duckdb.connect("data/duckdb_trains.db") as db:

    print("stations")
    display(db.sql("DESCRIBE services"))
    print("distances_long")
    display(db.sql("DESCRIBE distances_long"))
    print("distances_long")
    display(db.sql("DESCRIBE disruptions"))
    print("disruptions")
    display(db.sql("DESCRIBE stations"))
    print("stations")

# - services
# - distances_long
# - disruptions
# - stations

stations


┌──────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │ column_type │  null   │   key   │ default │  extra  │
│           varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Service:RDT-ID               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Date                 │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Type                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Company              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Train number         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Completely cancelled │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Partly cancelled     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ Service:Maximum delay        │ BIGINT    

distances_long


┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Station       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ other_station │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ distance      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

distances_long


┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ rdt_id               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ ns_lines             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rdt_lines            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rdt_lines_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rdt_station_names    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rdt_station_codes    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cause_nl             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cause_en             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ statistical_cause_nl │ VARCHAR     │ YES     │ NUL

disruptions


┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ code        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ uic         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ name_short  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ name_medium │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ name_long   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ slug        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ type        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ geo_lat     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ geo_lng     │ DOUB

stations


### 7. Make queries to answer the following questions:

In [40]:
# 7.1. How many trains departed from Amsterdam Central station overall?

with duckdb.connect("data/duckdb_trains.db") as db:

    amsterdam_stations = db.sql(
    """
    SELECT code, name_short, name_medium, name_long FROM stations
      WHERE "name_short" like 'Ams%'
        order by name_short
    """
    ).df()
    display(amsterdam_stations)

    ams_centr_code= amsterdam_stations[amsterdam_stations["name_long"] == "Amsterdam Centraal"]["code"].iloc[0]
    print(f"Amsterdam Centraal station code: {ams_centr_code=}")

    departed_from_amsterdam_central = db.execute(
    """
    SELECT COUNT(*) AS departed_from_amsterdam_central FROM services
        WHERE "Stop:Station code" = $ams_centr_code
          AND "Stop:Departure time" IS NOT NULL
    """,
    {"ams_centr_code": ams_centr_code}
    ).df()
    display(departed_from_amsterdam_central.head())





,code,name_short,name_medium,name_long
0,ASA,Amstel,Amstel,Amsterdam Amstel
1,ASD,Amsterdm C,Amsterdam C.,Amsterdam Centraal
2,ASDZ,Amsterdm Z,Amsterdam Zuid,Amsterdam Zuid
3,AMST,Amstetten,Amstetten NÖ,Amstetten NÖ


Amsterdam Centraal station code: ams_centr_code='ASD'


,departed_from_amsterdam_central
0,267099


7.2. Calculate the average arrival delay of different service types (Service:Type). Order results descending by average delay.

In [43]:
with duckdb.connect("data/duckdb_trains.db") as db:
    average_arrival_delay_by_service_type = db.sql(
    """
    SELECT
            "Service:Type",
            AVG("Stop:Arrival delay") AS avg_arrival_delay
        FROM services
        WHERE "Stop:Arrival delay" IS NOT NULL
        GROUP BY "Service:Type"
        ORDER BY avg_arrival_delay DESC
    """
    ).df()
    display(average_arrival_delay_by_service_type.head(10))

,Service:Type,avg_arrival_delay
0,European Sleeper,13.104823
1,Stoomtrein,12.000000
2,Nightjet,9.099789
3,Eurostar,7.788638
4,ICE International,6.904160
5,Int. Trein,4.480769
6,Nachttrein,3.885049
7,Intercity direct,3.252411
8,Extra trein,2.948206
9,Eurocity Direct,2.248236


7.3. What was the most common disruption cause in different years? MODE function may be useful.

In [51]:
with duckdb.connect("data/duckdb_trains.db") as db:

    # TODO: PROBLEM: we have data from 2023 only !!!
    display(db.execute("SELECT DISTINCT YEAR(start_time) AS year FROM disruptions ORDER BY year").df())
    # TODO: PROBLEM: we have data from 2024-25 only !!!
    display(db.execute("SELECT DISTINCT YEAR(\"Stop:Departure time\") AS year FROM services WHERE \"Stop:Departure time\" IS NOT NULL ORDER BY year").df())

    most_common_disruption_cause_by_year = db.sql(
    """
    SELECT YEAR("start_time") AS year, MODE("cause_en") AS most_common_cause
    FROM disruptions
        GROUP BY year
        ORDER BY year
    """
    ).df()
    display(most_common_disruption_cause_by_year.head())

,year
0,2023


,year
0,2024
1,2025


,year,most_common_cause
0,2023,broken down train


7.4. How many trains started their overall service in any Amsterdam station?

In [52]:
with duckdb.connect("data/duckdb_trains.db") as db:

    amsterdam_stations = db.sql(
    """
    SELECT code, name_short, name_medium, name_long FROM stations
      WHERE "name_short" like 'Ams%'
        order by name_short
    """
    ).df()
    display(amsterdam_stations)

    ams_station_codes = tuple(amsterdam_stations["code"].tolist())
    print(f"Amsterdam station codes: {ams_station_codes=}")

    trains_started_in_amsterdam = db.execute(
    """
    SELECT COUNT(DISTINCT "Service:RDT-ID") AS trains_started_in_amsterdam
        FROM services
        WHERE "Stop:Station code" IN $ams_station_codes
          AND "Stop:Arrival time" IS NULL
    """,
    {"ams_station_codes": ams_station_codes}
    ).df()
    display(trains_started_in_amsterdam.head())

,code,name_short,name_medium,name_long
0,ASA,Amstel,Amstel,Amsterdam Amstel
1,ASD,Amsterdm C,Amsterdam C.,Amsterdam Centraal
2,ASDZ,Amsterdm Z,Amsterdam Zuid,Amsterdam Zuid
3,AMST,Amstetten,Amstetten NÖ,Amstetten NÖ


Amsterdam station codes: ams_station_codes=('ASA', 'ASD', 'ASDZ', 'AMST')


,trains_started_in_amsterdam
0,148901


7.5. What fraction of services was run to final destinations outside the Netherlands?

In [54]:
with duckdb.connect("data/duckdb_trains.db") as db:
    # ----------------------------------------------
    # TODO: see comparison of null values in  homework_7_5_compare.py
    # ----------------------------------------------
    None



7.6. What is the largest distance between stations in the Netherlands (code NL)?

In [ ]:
# TODO:

7.7. Compare the average arrival delay between different train operators (Service:Company) on a bar plot. Sort them appropriately.

In [ ]:
# TODO:

7.8. How many services were disrupted in different years? Make a line plot.

In [ ]:
# TODO:

7.9. What fraction of all services were cancelled (Service:Completely cancelled) in different years? Make a line plot.

In [ ]:
# TODO: